# Inference — Sovereign Dialect-Bridge

```
INPUT teks (dialek daerah)
  -> detect_dialect   (TF-IDF + LogReg)
  -> translate to BI  (NLLB-200)
  -> summarize        (pilih: textrank | ner | indot5 | mt5)
  -> OUTPUT + ROUGE & CR (jika ada referensi)
```

## Konfigurasi

Ubah tiga variabel di bawah, lalu jalankan semua sel dari atas ke bawah.

In [6]:
INPUT_TEXT = """
Punten ieu téh kumaha kawijakan pamaréntah, hususna ti pihak PLN sareng Kementrian ESDM? Geus sababaraha poé ieu listrik di wewengkon Jawa, kaasup di lembur kuring di Jawa Barat, gawéna ngan pareum hurung, pareum hurung teu puguh aturan. Kacida pisan nganggu kana aktivitas sapopoé, komo deui loba parabot éléktronik di rorompok, sapertos kulkas sareng tivi, anu ruksak lantaran arus listrik nu teu stabil. Mun badé pareum téh tara aya béwara ti anggalna, ujug-ujug blang poék waé, terus hurung deui satengah jam, tuluy pareum deui. Ieu téh maén-maén atanapi kumaha?

Kuring meunang beja tina warta jeung ramé di média sosial, majar ieu pasualan téh gara-gara kawijakan ti Menteri ESDM, Pak Bahlil, anu konon cenah langkung ngutamakeun ékspor batubara ka luar nagri tibatan nyumponan pasokan pembangkit listrik di jero nagri. Naha enya pamaréntah téh leuwih mentingkeun kauntungan ti nagara deungeun batan ngalayanan rahayatna sorangan? Akibatna, suplai bahan bakar keur PLTU di pulo Jawa jadi ngurangan drastis, jeung ahirna PLN kapaksa ngalakukeun pamadaman bergilir lantaran defisit daya. Ieu téh kabijakan anu kacida pisan henteu adil keur masarakat leutik!

Sim kuring minangka palanggan anu taat, tara towong, jeung tara telat mayar tagihan listrik unggal bulan, nungtut sangkan ieu pasualan énggal dianggeuskeun. Ulah nepikeun ka rahayat anu jadi korban tina kawijakan élit di luhur nu teu pécus ngurus sumber daya alam nagara. Cik atuh Pak Bahlil sareng jajaran diréksi PLN, pék turun ka lapangan, rasakeun kumaha sangsarana mun unggal peuting kudu hareudang, poék-poékan, jeung kaganggu usaha. Lamun kieu waé carana, ulah nyalahkeun mun engké masarakat sapulo Jawa mogok embung mayar tagihan listrik sasih payun!
""".strip()

# Pilih metode summarisasi: "textrank, ner, indot5, mt5"
# textrank dan ner = extractive (cepat, tanpa model neural besar)
# indot5 = abstractive ringan (~853 MB), mt5 = abstractive berat (~2.2 GB)
SUMMARIZE_WITH = "textrank, ner, indot5"

# Opsional: isi teks referensi untuk hitung ROUGE & CR.
# Kosongkan jika tidak punya referensi.
REFERENCE_TEXT = """
Pelanggan menyampaikan komplain keras dan penuh amarah terkait kondisi listrik di wilayah Jawa (khususnya Jawa Barat) yang terus-menerus mengalami byar-pet (mati-nyala) tanpa pemberitahuan. Kondisi ini telah merusak peralatan elektronik warga dan mengganggu aktivitas. Pelanggan menyoroti bahwa penyebab utama pemadaman ini adalah defisit daya listrik akibat kebijakan Menteri ESDM (Bahlil) yang diduga lebih memprioritaskan ekspor batubara ke luar negeri daripada memasok kebutuhan PLTU domestik di Pulau Jawa. Warga merasa sangat dirugikan karena mereka selalu tertib membayar tagihan listrik, dan mengancam akan melakukan mogok bayar tagihan bulan depan jika pemerintah dan PLN tidak segera menyelesaikan krisis energi ini.
""".strip()

## 1. Setup

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import time
from pathlib import Path

import joblib
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Device: {DEVICE}")

_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebook" else _cwd

DIALECT_MODEL_PATH = PROJECT_ROOT / "notebook" / "dialect_detector" / "dialect_detector.joblib"
print(f"Dialect model : {'OK' if DIALECT_MODEL_PATH.exists() else 'MISSING — jalankan train_dialect_detector.ipynb dulu'}")

/opt/homebrew/Caskroom/miniforge/base/envs/ai_core/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
Dialect model : OK


## 2. Detect Dialect

In [3]:
DIALECT_NAMES = {
    "id": "Bahasa Indonesia", "jv": "Bahasa Jawa",
    "su": "Bahasa Sunda",     "min": "Minangkabau",
    "ace": "Aceh",            "ban": "Bali",
    "bjn": "Banjar",          "bug": "Bugis",
    "mad": "Madura",          "nij": "Ngaju Dayak",
    "bbc": "Batak Toba",      "en": "English",
    "xx": "Tidak Terdeteksi",
}

MIN_DIALECT_CONF = 0.35

dialect_clf = joblib.load(DIALECT_MODEL_PATH)


def detect_dialect(text: str) -> tuple[str, float]:
    if len(text.strip()) < 10:
        return "xx", 0.0
    probs   = dialect_clf.predict_proba([text.strip()])[0]
    classes = list(dialect_clf.classes_)
    top_idx = int(probs.argmax())
    label, conf = classes[top_idx], float(probs[top_idx])
    return (label, round(conf, 3)) if conf >= MIN_DIALECT_CONF else ("xx", round(conf, 3))


dialect, dialect_conf = detect_dialect(INPUT_TEXT)
print(f"Dialek   : {dialect} — {DIALECT_NAMES.get(dialect, dialect)}")
print(f"Confidence: {dialect_conf:.3f}")

Dialek   : su — Bahasa Sunda
Confidence: 0.980


## 3. Translate ke Bahasa Indonesia (NLLB-200)

In [4]:
DIALECT_TO_NLLB = {
    "id": "ind_Latn", "jv": "jav_Latn", "su": "sun_Latn",
    "min": "min_Latn", "ace": "ace_Latn", "ban": "ban_Latn",
    "bjn": "bjn_Latn", "bug": "bug_Latn", "en": "eng_Latn",
}
TARGET_LANG = "ind_Latn"
NLLB_ID     = "facebook/nllb-200-distilled-600M"


def _translate_deep(text: str) -> str:
    try:
        from deep_translator import GoogleTranslator
        out = GoogleTranslator(source="auto", target="id").translate(text[:4500])
        return out if out and out.strip() else text
    except Exception:
        return text


def translate_to_indonesian(text: str, src_dialect: str,
                             nllb_tok, nllb_mdl) -> str:
    if src_dialect in ("id", "xx"):
        return text
    nllb_lang = DIALECT_TO_NLLB.get(src_dialect)
    if not nllb_lang:
        return _translate_deep(text)
    nllb_tok.src_lang = nllb_lang
    inputs    = nllb_tok(text[:1000], return_tensors="pt", truncation=True,
                         max_length=512).to(DEVICE)
    target_id = nllb_tok.convert_tokens_to_ids(TARGET_LANG)
    with torch.no_grad():
        out = nllb_mdl.generate(**inputs, forced_bos_token_id=target_id,
                                max_new_tokens=256, num_beams=2)
    return nllb_tok.decode(out[0], skip_special_tokens=True)


print(f"Loading NLLB ({NLLB_ID})...")
t0 = time.time()
nllb_tok = AutoTokenizer.from_pretrained(NLLB_ID)
nllb_mdl = AutoModelForSeq2SeqLM.from_pretrained(NLLB_ID).to(DEVICE).eval()
print(f"  loaded ({time.time()-t0:.1f}s)")

t0 = time.time()
translated = translate_to_indonesian(INPUT_TEXT, dialect, nllb_tok, nllb_mdl)
print(f"\nTerjemahan ({time.time()-t0:.1f}s):")
print(f"  {translated}")

Loading NLLB (facebook/nllb-200-distilled-600M)...


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0).
W0621 14:36:43.786000 41774 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 512/512 [00:00<00:00, 32408.98it/s]


  loaded (38.9s)


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Terjemahan (33.9s):
  Apakah ini kebijakan pemerintah, terutama PLN dan Kementrian ESDM? Beberapa hari ini listrik di Jawa, termasuk di tempat kerja saya di Jawa Barat, toko dan pareum hurung, pareum hurung tidak lagi regulated. Tidak banyak aktivitas sehari-hari, apalagi banyak alat elektronik di grup, seperti kulkas dan televisi, yang rusak karena arus listrik yang tidak stabil. Apakah pareum hanya menghangatkan air, tiba-tiba blang, terus menghangatkan air selama setengah jam, dan kemudian pareum lagi. Apakah ini bermain-main atau tidak? Saya mendapat keuntungan dari media sosial, tetapi ini adalah langkah-langkah dari Menteri ESDM, Bahlil, Pakatan Camat, dan sebagainya.


## 4. Summarize

Hanya model yang dipilih di `SUMMARIZE_WITH` yang di-load.

In [8]:
def _split_sentences(text: str) -> list[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]


def _first_sentences(text: str, n: int = 2) -> str:
    return " ".join(_split_sentences(text)[:n]) or text.strip()


def summarize_textrank(text: str, n_sents: int = 3, max_words: int = 80) -> str:
    import networkx as nx
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    sents = _split_sentences(text)
    if len(sents) <= n_sents:
        result = " ".join(sents)
    else:
        mat    = TfidfVectorizer().fit_transform(sents)
        sim    = cosine_similarity(mat, mat)
        np.fill_diagonal(sim, 0)
        scores = nx.pagerank(nx.from_numpy_array(sim))
        ranked = sorted(scores, key=scores.get, reverse=True)[:n_sents]
        result = " ".join(sents[i] for i in sorted(ranked))
    words = result.split()
    return " ".join(words[:max_words]) if len(words) > max_words else result


def summarize_ner(text: str, n_sents: int = 3, max_words: int = 80) -> str:
    from transformers import pipeline as hf_pipeline
    ner    = hf_pipeline("ner", model="cahya/bert-base-indonesian-NER",
                          aggregation_strategy="simple", device=-1)
    sents  = _split_sentences(text)
    if len(sents) <= n_sents:
        result = " ".join(sents)
    else:
        ents   = {e["word"].lower() for e in ner(text[:512]) if e.get("score", 0) > 0.5}
        scores = [sum(1 for w in ents if w in s.lower()) for s in sents]
        ranked = sorted(range(len(sents)), key=lambda i: -scores[i])[:n_sents]
        result = " ".join(sents[i] for i in sorted(ranked))
    words = result.split()
    return " ".join(words[:max_words]) if len(words) > max_words else result


def _abstractive(text: str, model_id: str, prefix: str, min_words: int = 40) -> str:
    if len(text.split()) < min_words:
        return _first_sentences(text)
    print(f"  Loading {model_id}...")
    t0  = time.time()
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(DEVICE).eval()
    print(f"  loaded ({time.time()-t0:.1f}s)")
    enc = tok(prefix + text[:1024], return_tensors="pt",
              truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = mdl.generate(**enc, max_new_tokens=150, num_beams=4,
                           no_repeat_ngram_size=3, early_stopping=True,
                           length_penalty=1.0)
    result = tok.decode(out[0], skip_special_tokens=True)
    result = re.sub(r"([a-z])([A-Z])", r"\1 \2", result)
    result = re.sub(r"\s+", " ", result).strip()
    return result[0].upper() + result[1:] if result else ""


ABSTRACTIVE_IDS = {
    "indot5": ("OinoVenv/sovereign-indot5-nusasum", "ringkas: "),
    "mt5":    ("OinoVenv/sovereign-mt5-nusasum",    "summarize: "),
}

VALID_METHODS = {"textrank", "ner", "indot5", "mt5"}


def run_summarize(text: str, method: str) -> str:
    if method == "textrank":
        return summarize_textrank(text)
    if method == "ner":
        return summarize_ner(text)
    if method in ABSTRACTIVE_IDS:
        model_id, prefix = ABSTRACTIVE_IDS[method]
        return _abstractive(text, model_id, prefix)
    raise ValueError(f"Method tidak dikenal: {method!r}. Pilih: textrank, ner, indot5, mt5")


# Parse SUMMARIZE_WITH: bisa satu method ("indot5") atau beberapa ("textrank, ner, indot5")
selected_methods = [m.strip() for m in SUMMARIZE_WITH.split(",") if m.strip()]
invalid = [m for m in selected_methods if m not in VALID_METHODS]
if invalid:
    raise ValueError(f"Method tidak dikenal: {invalid}. Pilih dari: {sorted(VALID_METHODS)}")

summaries = {}   # method -> (summary_text, elapsed_ms)
for method in selected_methods:
    print(f"\n[{method}] running...")
    t0 = time.time()
    summaries[method] = (run_summarize(translated, method),
                         round((time.time() - t0) * 1000))
    print(f"  done ({summaries[method][1]} ms)")


[textrank] running...
  done (460 ms)

[ner] running...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 32478.56it/s]
BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  done (3276 ms)

[indot5] running...
  Loading OinoVenv/sovereign-indot5-nusasum...


Loading weights: 100%|██████████| 257/257 [00:00<00:00, 4162.19it/s]


  loaded (19.2s)
  done (58790 ms)


## 5. Hasil & Evaluasi

In [9]:
from rouge_score import rouge_scorer as _rouge_lib

scorer = _rouge_lib.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False) \
         if REFERENCE_TEXT.strip() else None

sep = "=" * 70
print(sep)
print(f"INPUT ({len(INPUT_TEXT.split())} kata):")
print(f"  {INPUT_TEXT[:300]}{'...' if len(INPUT_TEXT) > 300 else ''}")
print()
print(f"DIALEK     : {dialect} — {DIALECT_NAMES.get(dialect, dialect)}  (conf={dialect_conf:.3f})")
print(f"TERJEMAHAN : {translated}")
print(sep)

for method, (summary, elapsed_ms) in summaries.items():
    n_in  = len(translated.split())
    n_out = len(summary.split())
    cr    = round(n_out / n_in, 3) if n_in > 0 else 0.0

    print(f"\nMETODE    : {method}  ({elapsed_ms} ms)")
    print(f"RINGKASAN : {summary}")
    print(f"CR        : {cr}  ({n_out} kata / {n_in} kata input terjemahan)")

    if scorer:
        scores = scorer.score(REFERENCE_TEXT.strip(), summary)
        print(f"ROUGE-1   : {scores['rouge1'].fmeasure:.4f}")
        print(f"ROUGE-2   : {scores['rouge2'].fmeasure:.4f}")
        print(f"ROUGE-L   : {scores['rougeL'].fmeasure:.4f}")

if REFERENCE_TEXT.strip():
    print(f"\n{sep}")
    print(f"REFERENSI : {REFERENCE_TEXT.strip()}")
else:
    print(f"\n(Isi REFERENCE_TEXT di cell konfigurasi untuk menghitung ROUGE.)")

INPUT (258 kata):
  Punten ieu téh kumaha kawijakan pamaréntah, hususna ti pihak PLN sareng Kementrian ESDM? Geus sababaraha poé ieu listrik di wewengkon Jawa, kaasup di lembur kuring di Jawa Barat, gawéna ngan pareum hurung, pareum hurung teu puguh aturan. Kacida pisan nganggu kana aktivitas sapopoé, komo deui loba pa...

DIALEK     : su — Bahasa Sunda  (conf=0.980)
TERJEMAHAN : Apakah ini kebijakan pemerintah, terutama PLN dan Kementrian ESDM? Beberapa hari ini listrik di Jawa, termasuk di tempat kerja saya di Jawa Barat, toko dan pareum hurung, pareum hurung tidak lagi regulated. Tidak banyak aktivitas sehari-hari, apalagi banyak alat elektronik di grup, seperti kulkas dan televisi, yang rusak karena arus listrik yang tidak stabil. Apakah pareum hanya menghangatkan air, tiba-tiba blang, terus menghangatkan air selama setengah jam, dan kemudian pareum lagi. Apakah ini bermain-main atau tidak? Saya mendapat keuntungan dari media sosial, tetapi ini adalah langkah-langkah dari Menteri E